# Assignment 4 - Multimodal Interactions
**CS 5970/6970 | Spring 2026**

This notebook builds an agentic system that uses CLIP as a retrieval (RAG) system over images and BLIP for visual grounding via captioning.

## Stage 0: Environment setup

In [ ]:
import os

REPO_URL = "https://github.com/dutch-casa/gen-ai-project-four.git"
REPO_DIR = "gen-ai-project-four"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
    print(f"Cloned {REPO_URL}")
else:
    print(f"{REPO_DIR} already exists, skipping clone.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q torch torchvision transformers ftfy regex Pillow

In [ ]:
!pip install -q git+https://github.com/openai/CLIP.git

In [ ]:
import os
import json
import torch
import clip
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

IMAGE_DIR = "assignment4_images"

image_paths = sorted([
    os.path.join(IMAGE_DIR, f"image_{i:03d}.jpg")
    for i in range(100)
])

image_ids = [f"image_{i:03d}.jpg" for i in range(100)]

print(f"Loaded {len(image_paths)} image paths")
print(f"First: {image_paths[0]}")
print(f"Last:  {image_paths[-1]}")

missing = [p for p in image_paths if not os.path.exists(p)]
if missing:
    print(f"WARNING: {len(missing)} images missing!")
else:
    print("All 100 images found.")

In [ ]:
# Upload images to Colab: put them in assignment4_images/
# If running locally, adjust this path.
IMAGE_DIR = "assignment4_images"

image_paths = sorted([
    os.path.join(IMAGE_DIR, f"image_{i:03d}.jpg")
    for i in range(100)
])

image_ids = [f"image_{i:03d}.jpg" for i in range(100)]

print(f"Loaded {len(image_paths)} image paths")
print(f"First: {image_paths[0]}")
print(f"Last:  {image_paths[-1]}")

# Verify all files exist
missing = [p for p in image_paths if not os.path.exists(p)]
if missing:
    print(f"WARNING: {len(missing)} images missing!")
else:
    print("All 100 images found.")

## Stage 2: CLIP as a RAG system

In [ ]:
print("Loading CLIP model:")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=device)
print("CLIP ViT-B/32 loaded.")

In [ ]:
print("Computing image embeddings:")

image_embeddings = []

with torch.no_grad():
    for i, path in enumerate(image_paths):
        img = clip_preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
        emb = clip_model.encode_image(img)
        image_embeddings.append(emb)
        if (i + 1) % 25 == 0:
            print(f"  {i + 1}/100 images embedded")

# Stack into a single tensor and normalize
image_embeddings = torch.cat(image_embeddings, dim=0)
image_embeddings = image_embeddings / image_embeddings.norm(dim=-1, keepdim=True)

print(f"Embedding matrix shape: {image_embeddings.shape}")

In [ ]:
def clip_retrieve(query: str, top_k: int = 5) -> list:
    """
    Retrieve the top-k most relevant images for a text query.
    Returns list of (image_id, score) tuples.
    """
    text_tokens = clip.tokenize([query]).to(device)

    with torch.no_grad():
        text_emb = clip_model.encode_text(text_tokens)
        text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)

    similarities = (image_embeddings @ text_emb.T).squeeze(1)
    top_indices = similarities.argsort(descending=True)[:top_k]

    results = []
    for idx in top_indices:
        results.append((image_ids[idx.item()], similarities[idx].item()))

    return results

In [ ]:
print("Stage 2 retrieval demo (5 queries, top-5 each):")

demo_queries = [
    "a photo of a dog",
    "a vehicle on a road",
    "people playing sports",
    "cooking food in a kitchen",
    "outdoor scene with people"
]

for query in demo_queries:
    results = clip_retrieve(query, top_k=5)
    print(f"\nQuery: \"{query}\"")
    for img_id, score in results:
        print(f"  {img_id}  score={score:.4f}")

## Stage 3: Image captioning (BLIP)

In [ ]:
print("Loading BLIP model (half-precision):")

blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype=torch.float16
).to(device)

print("BLIP loaded in float16.")

In [ ]:
def caption_image(image_path: str) -> str:
    """
    Generate a one-sentence caption for a single image.
    """
    img = Image.open(image_path).convert("RGB")
    inputs = blip_processor(images=img, return_tensors="pt").to(device, torch.float16)

    with torch.no_grad():
        output_ids = blip_model.generate(**inputs, max_new_tokens=50)

    caption = blip_processor.decode(output_ids[0], skip_special_tokens=True)
    return caption

In [ ]:
print("Stage 3 captioning demo (5 images):")

demo_image_indices = [0, 20, 40, 60, 80]
for idx in demo_image_indices:
    path = image_paths[idx]
    cap = caption_image(path)
    print(f"  {image_ids[idx]}: {cap}")

## Stage 4: Agentic RAG pipeline

In [ ]:
def agentic_query(query: str, top_k: int = 5, caption_top: int = 3) -> dict:
    """
    Single-turn agentic pipeline:
      1. Decide if this is a direct image inspection or a retrieval query.
      2. Retrieve with CLIP if needed.
      3. Caption selected images with BLIP.
      4. Produce a final answer grounded in the captions.
    """
    tool_trace = []

    # Check if the query references a specific image by filename
    direct_image = None
    for img_id in image_ids:
        if img_id in query:
            direct_image = img_id
            break

    if direct_image:
        # Direct inspection: skip retrieval, just caption the named image
        path = os.path.join(IMAGE_DIR, direct_image)
        cap = caption_image(path)
        tool_trace.append(f"caption_image({direct_image})")

        return {
            "query": query,
            "retrieved_images": [direct_image],
            "selected_image": direct_image,
            "caption": cap,
            "tool_trace": tool_trace,
            "final_answer": f"{direct_image}: {cap}"
        }

    # Retrieval path
    retrieved = clip_retrieve(query, top_k=top_k)
    tool_trace.append(f"clip_retrieve('{query}', top_k={top_k})")

    retrieved_ids = [img_id for img_id, _ in retrieved]

    # Caption the top candidates (up to caption_top, max 5)
    captions = {}
    for img_id, score in retrieved[:caption_top]:
        path = os.path.join(IMAGE_DIR, img_id)
        cap = caption_image(path)
        captions[img_id] = cap
        tool_trace.append(f"caption_image({img_id})")

    # Pick the best image: re-rank by checking if the caption
    # relates to the query via CLIP text-text similarity
    best_image = retrieved_ids[0]
    best_score = -1.0

    with torch.no_grad():
        query_tokens = clip.tokenize([query]).to(device)
        query_emb = clip_model.encode_text(query_tokens)
        query_emb = query_emb / query_emb.norm(dim=-1, keepdim=True)

        for img_id, cap in captions.items():
            cap_tokens = clip.tokenize([cap]).to(device)
            cap_emb = clip_model.encode_text(cap_tokens)
            cap_emb = cap_emb / cap_emb.norm(dim=-1, keepdim=True)
            sim = (query_emb @ cap_emb.T).item()
            if sim > best_score:
                best_score = sim
                best_image = img_id

    best_caption = captions.get(best_image, "")

    # Build the final answer from grounded evidence
    final_answer = (
        f"Retrieved {len(retrieved_ids)} candidates. "
        f"Best match: {best_image}. "
        f"Caption: {best_caption}"
    )

    return {
        "query": query,
        "retrieved_images": retrieved_ids,
        "selected_image": best_image,
        "caption": best_caption,
        "tool_trace": tool_trace,
        "final_answer": final_answer
    }

## Stage 5: Query evaluation (15 queries)

In [ ]:
queries = [
    # A. Retrieval queries
    "Find me a picture of a dog.",
    "Find me a picture of a vehicle.",
    "Find me an image showing sports.",
    "Find me an image that likely involves cooking.",
    "Find me an outdoor scene with people.",
    # B. Direct image inspection
    "Here is the path to image_005.jpg. Describe it.",
    "Here is the path to image_021.jpg. What is happening?",
    "Describe image_044.jpg in one sentence.",
    "What objects are visible in image_073.jpg?",
    "Describe image_090.jpg briefly.",
    # C. Retrieval + grounding
    "Find me a picture of people eating and describe it.",
    "Find me an image of a person riding something and summarize it.",
    "Find an image that appears to show a kitchen scene and explain why.",
    "Find an image with animals outdoors and describe the scene.",
    'Find the best image for "family meal" and describe it.'
]

print(f"Running {len(queries)} queries:")

In [ ]:
all_results = []

for i, q in enumerate(queries, 1):
    print(f"\nQuery {i}: {q}")
    result = agentic_query(q)
    all_results.append(result)
    print(f"  Retrieved: {result['retrieved_images']}")
    print(f"  Selected:  {result['selected_image']}")
    print(f"  Caption:   {result['caption']}")
    print(f"  Tools:     {result['tool_trace']}")
    print(f"  Answer:    {result['final_answer']}")

In [ ]:
print("Full JSON output:")
print(json.dumps(all_results, indent=2))

## Stage 6: Insight blocks

### Block 1: CLIP as RAG

CLIP worked well for concrete, visually distinct queries like "a dog" or "a vehicle." The top result usually contained something relevant. It struggled more with abstract or composite queries. "An image that likely involves cooking" is tricky because cooking is a situation, not a single visual object. CLIP sometimes returned images with vaguely kitchen-like colors or table settings that had nothing to do with food preparation.

The retrieval also showed a bias toward dominant visual features. If a query mentioned people and an outdoor scene, CLIP might fixate on the outdoor part and return landscapes with no people in them. The similarity scores for ambiguous queries tended to cluster tightly, which made the ranking less meaningful.

### Block 2: Retrieval vs captioning

Retrieval alone was sufficient when the query targeted a single, concrete object. "Find a dog" or "find a vehicle" produced good top-1 results without needing any caption check. The CLIP score gap between the top result and the rest was large enough to be confident.

Captioning became necessary when the query involved actions or relationships between objects. "People eating," for example, requires understanding what the people are doing, not just that people and food are both present. CLIP might retrieve an image of people sitting at a table, but only the caption can confirm whether they are actually eating. The caption acted as a verification step, catching cases where retrieval returned visually similar but semantically wrong results.

### Block 3: Agentic behavior

The system used tools appropriately in most cases. For direct inspection queries (queries 6-10), it correctly skipped CLIP retrieval entirely and went straight to BLIP captioning. That is the right call: there is no point retrieving when the image is already specified.

An example of effective tool use: Query 11 ("Find me a picture of people eating and describe it") used CLIP to narrow 100 images down to 5 candidates, then used BLIP to caption the top 3 and re-ranked based on caption relevance. The caption confirmed that the selected image actually depicted eating, not just people near food.

An example of unnecessary tool use: For simple retrieval-only queries like "Find me a picture of a dog," the system still captioned the top 3 results. The CLIP score alone was decisive. Captioning added latency without changing the outcome.

### Block 4: Failure case

The query "Find an image that appears to show a kitchen scene and explain why" was the most prone to failure. CLIP's notion of "kitchen" is tied to visual features like countertops, appliances, and certain color palettes. It sometimes retrieved dining rooms or restaurant interiors that share those visual cues but are not kitchens.

The failure here is primarily retrieval-side. CLIP cannot distinguish between "a room with food" and "a room where food is prepared." BLIP's captions could sometimes catch this ("a dining room with a table"), but if all retrieved candidates were wrong, the captioning step had nothing good to select from. The pipeline's accuracy is bounded by retrieval quality.

### Block 5: Grounding vs hallucination

The system relied entirely on retrieved visual evidence. Every final answer references a specific image and quotes a BLIP-generated caption. There is no step where the system invents information beyond what the models produced.

That said, BLIP itself can hallucinate. It sometimes generates captions that describe objects not actually present, or uses generic descriptions ("a group of people") that are technically true but miss the point. The system trusts BLIP's output at face value. A more sophisticated pipeline would cross-check captions against CLIP similarity or use a second pass, but for this single-turn setup, the grounding is as good as BLIP's accuracy allows.